In [ ]:
"""
Updated on Fri Oct 31 
@author: Jingyi 
Created on Frid Jan 21 10:02:01 2022
@author: Oumbeg
"""
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import numpy as np
from pandas import ExcelWriter
from selenium import webdriver
from time import sleep
import os
import re
import requests
import tabula
import camelot
from docx import Document

# from docx2pdf import convert
from openpyxl import load_workbook

In [162]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'UG BUG' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Read {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])


scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


Running UG BUG Web Scraping Tool v.1.0


In [175]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

In [215]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        'UG BUG 1':'Commercial Banks' , 

        'UG BUG 2': 'Credit Institutions', 

        'UG BUG 3': 'Forex Bureaux',

        'UG BUG 4': 'Money Remitters', 

        'UG BUG 5': 'Microfinance Deposit-taking Institutions', 


        }


Typology={

        'UG BUG 1':'Commercial Banks' , 

        'UG BUG 2': 'Credit Institutions', 

        'UG BUG 3': 'Forex Bureaux',

        'UG BUG 4': 'Money Remitters', 

        'UG BUG 5': 'Microfinance Deposit-taking Institutions', 


        }


sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}


processdate = now.strftime('%Y-%m-%d')





In [165]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


def cell_text(cell):
    return " ".join(
        p.text.strip()
        for p in cell.paragraphs
        if p.text.strip()
    )


In [173]:
mainURL  =  'https://bou.or.ug/bouwebsite/Supervision/supervisedinstitutions.html'
response = requests.get(mainURL, verify=False, timeout=30)

data = response.text  
soup = BeautifulSoup(data, "html.parser")

table = soup.find("div",{"class":"contenttext"})
typology = table.find_all("li")


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'bou.or.ug'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
for reg in regdict:
    print(f"Processing {reg}...")

    resp = requests.get(mainURL, verify=False, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    section_name = regdict[reg]  # Money Remitters
    link = soup.select_one(f"div.contenttext li a:-soup-contains('{section_name}')")
    if not link:
        raise ValueError(f"Could not find link containing '{section_name}'")

    href = link.get("href")
    print("Found link:", href)

    driver.get(href)
    # …process next_resp.content or save the file

    if reg == 'UG BUG 1':
        sleep(5)

        doc = Document(filePath)
        row_data = []
        for table in doc.tables:
            
            for row in table.rows[:]:
                num = cell_text(row.cells[0])
                name = cell_text(row.cells[1])
                addrs = cell_text(row.cells[2])
                phone_ = cell_text(row.cells[3])
                swift_code = cell_text(row.cells[4])

                row_data.append((num, name, addrs,phone_,swift_code))

        consolidated = []
        current = None

        for row in row_data:
            processed = []
            for value in row:
                if value is None:
                    processed.append("")
                elif isinstance(value, list):
                    bits = [str(part).strip() for part in value if str(part).strip()]
                    processed.append(" ".join(bits))
                else:
                    processed.append(str(value).strip())
            idx, name, addr, phone, swift = processed

            if re.match(r"^\d+", idx):
                if current is not None:
                    consolidated.append(
                        (
                            current["no"],
                            current["name"],
                            " ".join(current["address"]).strip(),
                            " ".join(current["phone"]).strip(),
                            " ".join(current["swift"]).strip(),
                        )
                    )
                current = {
                    "no": idx.rstrip("."),
                    "name": name,
                    "address": [addr] if addr else [],
                    "phone": [phone] if phone else [],
                    "swift": [swift] if swift else [],
                }
            elif current is not None:
                if name:
                    current["name"] = f"{current['name']} {name}".strip()
                if addr:
                    current["address"].append(addr)
                if phone:
                    current["phone"].append(phone)
                if swift:
                    current["swift"].append(swift)

        if current is not None:
            consolidated.append(
                (
                    current["no"],
                    current["name"],
                    " ".join(current["address"]).strip(),
                    " ".join(current["phone"]).strip(),
                    " ".join(current["swift"]).strip(),
                )
            )

        for rec in consolidated:
            clean_name = rec[1].strip()
            tokens_name = clean_name.split()
            half_name = len(tokens_name) // 2

            if len(tokens_name) % 2 == 0 and tokens_name[:half_name] == tokens_name[half_name:]:
                clean_name = " ".join(tokens_name[:half_name])
                sqldict['Name'].append(clean_name)
                sqldict['ListProcessDate'].append(processdate)
            else:
                sqldict['Name'].append(clean_name)
                sqldict['ListProcessDate'].append(processdate)

            sqldict['ListProcessDate'].append(processdate)
            sqldict['Address_1'].append(rec[2])
            sqldict['Phone'].append(rec[3])
            #sqldict['BIC SWIFT Code'].append(rec[4])
            sqldict['RegCtry'].append('UG')
            sqldict['Cntry'].append('UG')
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['RegulationType'].append('Regulated') 
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        if os.path.exists(tempfolder):
            for temp_file in os.listdir(tempfolder):
                os.remove(os.path.join(tempfolder, temp_file))
        else:
            os.mkdir(tempfolder)
    elif reg == 'UG BUG 2' or reg == 'UG BUG 5':
            sleep(10)
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        


            sleep(3)
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')
            # Iterate over the tables of each pages
            for i in range(tables.n):
                df_Table = tables[i].df
        
                for j in range(len(df_Table)):
                    if df_Table[1][j].strip() not in ['','NAME'] :

                        sqldict['Name'].append(' '.join([item.strip() for item in df_Table[1][j].splitlines()]).strip())
                        sqldict['Address_1'].append(' '.join([item.strip() for item in df_Table[2][j].splitlines()]).strip())
                        sqldict['RegCtry'].append('UG')
                        sqldict['Cntry'].append('UG')
                        sqldict['Phone'].append(' '.join([item.strip() for item in df_Table[3][j].splitlines()]).replace('+','').replace('256','+256').strip())
                        if reg == 'UG BUG 2':

                            sqldict['BIC SWIFT Code'].append(' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip())
                        else:
                            if len(' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip())<3:
                                sqldict['Fax'].append('')
                            else:
                                sqldict['Fax'].append(' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip())
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict['RegCode'].append('BUG')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['ListName'].append(Typology[reg])
                    sqldict = bourange_same_length_array(sqldict)

            if os.path.exists(tempfolder):
                for temp_file in os.listdir(tempfolder):
                    os.remove(os.path.join(tempfolder, temp_file))
            else:
                os.mkdir(tempfolder)
    elif reg ==  'UG BUG 3':
        sleep(10)
        pdf_file = os.listdir(tempfolder)[0]
        filePath = os.path.join(tempfolder, pdf_file)

        print(filePath)
        tables_excel = pd.read_excel(filePath)
        first_col = tables_excel.columns[1]
        tables_excel = tables_excel[tables_excel[first_col].notna()]
        tables_excel.columns = tables_excel.iloc[0]
        tables_excel = tables_excel[1:]

        second_last_col = tables_excel.columns[-2]
        last_col = tables_excel.columns[-1]
        print(tables_excel.shape)

        for _, row in tables_excel.iterrows():
            name = row[second_last_col]
            address = row[last_col]
            # use the two values
            #print(name, address)
            sqldict['Name'].append(name)
            sqldict['Address_1'].append(address)
            sqldict['RegCtry'].append('UG')
            sqldict['Cntry'].append('UG')
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['RegCode'].append('BUG')
            sqldict['RegulationType'].append('Regulated') 
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        os.remove(filePath)
    elif  reg ==  'UG BUG 4':
        sleep(10)
        pdf_file = os.listdir(tempfolder)[0]
        filePath = os.path.join(tempfolder, pdf_file)
        clean_rows = []
        wb = load_workbook(filePath, data_only=True)
        ws = wb.active

        for row in ws.iter_rows(min_row=2, values_only=True):
            if all(v in (None, "") for v in row):
                continue                    # drop empty rows
            
            if row[0] is None:               # continuation row (only the address has text)
                if clean_rows:
                    clean_rows[-1][2] = (
                        (clean_rows[-1][2] or "") + "\n" + (row[2] or "")
                    )
                continue                     # don't keep this row by itself
            
            clean_rows.append([row[0], row[1], row[2], row[3]])
        for rwo in clean_rows:
            if isinstance(rwo[0], int):
                # print safe representations (avoid None.replace error)
                name = rwo[1]
                address_1 = rwo[2]
                # normalize address_2: remove newlines if it's a string, otherwise empty
                address_2 = rwo[3].replace('\n', '') if isinstance(rwo[3], str) else ''
                sqldict['Name'].append(name)
                sqldict['Address_1'].append(address_1)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append('UG')
                sqldict['Cntry'].append('UG')
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict['RegulationType'].append('Regulated') 
                sqldict['ListName'].append(Typology[reg])
        os.remove(filePath)
        


Processing UG BUG 1...
Found link: https://bou.or.ug/bouwebsite/bouwebsitecontent/Supervision/Supervised_Institutions/Supervision/financial_institutions/2025/LICENSED-COMMERCIAL-BANKS-AS-AT-3-MARCH-2025.docx


In [207]:
df=pd.DataFrame(sqldict)
writer = ExcelWriter(filename)
df = df[df['Name']!='']
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()

sleep(3)

driver.quit()

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_28596\3339668605.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [204]:
df.to_excel(filename,index=False)

In [212]:
df.to_csv('list1.csv')